Here we are starting the analysis of full dataset and there can be changes 

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
DATA_PATH = Path("../data/raw/paysim.csv")

In [5]:
data_types = {
    "step": "int16",
    "type": "category",
    "amount": "float64",
    "nameOrig": "string",
    "oldbalanceOrg": "float64",
    "newbalanceOrig": "float64",
    "nameDest": "string",
    "oldbalanceDest": "float64",
    "newbalanceDest": "float64",
    "isFraud": "int8",
    "isFlaggedFraud": "int8"
}

df = pd.read_csv(DATA_PATH, dtype=data_types)

In [6]:
df.shape

(6362620, 11)

Checking for quality entire dataset

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype   
---  ------          -----   
 0   step            int16   
 1   type            category
 2   amount          float64 
 3   nameOrig        string  
 4   oldbalanceOrg   float64 
 5   newbalanceOrig  float64 
 6   nameDest        string  
 7   oldbalanceDest  float64 
 8   newbalanceDest  float64 
 9   isFraud         int8    
 10  isFlaggedFraud  int8    
dtypes: category(1), float64(5), int16(1), int8(2), string(2)
memory usage: 370.1 MB


In [8]:
df.isna().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df["isFraud"].value_counts(normalize=True).mul(100)

isFraud
0    99.870918
1     0.129082
Name: proportion, dtype: float64

In [11]:
fraud_by_type = (
    df.groupby("type", observed=True)
    .agg(
        transactions_count=("isFraud", "size"),
        fraud_count=("isFraud", "sum"),
        fraud_rate=("isFraud", "mean")
    )
)

fraud_by_type["fraud_rate_percent"] = (
    fraud_by_type["fraud_rate"] * 100
).round(4)

fraud_by_type = fraud_by_type.sort_values(
    "fraud_rate_percent",
    ascending=False
)

fraud_by_type

,transactions_count,fraud_count,fraud_rate,fraud_rate_percent
type,,,,
TRANSFER,532909,4097,0.007688,0.7688
CASH_OUT,2237500,4116,0.001840,0.1840
CASH_IN,1399284,0,0.000000,0.0000
DEBIT,41432,0,0.000000,0.0000
PAYMENT,2151495,0,0.000000,0.0000


There we could find out that fraud operations correlates with two types of operations

In [12]:
risk_df = df[df["type"].isin(["TRANSFER", "CASH_OUT"])].copy()
risk_df.shape

(2770409, 11)

In [13]:
risk_df["amount_to_balance_ratio"] = np.where(risk_df["oldbalanceOrg"]>0, risk_df["amount"]/risk_df["oldbalanceOrg"], np.nan)

In [14]:
risk_df["emptied_origin_account"] = (
    (risk_df["oldbalanceOrg"] > 0) &
    (risk_df["newbalanceOrig"] == 0)
).astype("int8")

In [15]:
risk_df["origin_balance_error"] = (
    risk_df["oldbalanceOrg"]
    - risk_df["amount"]
    - risk_df["newbalanceOrig"]
).abs()

In [16]:
risk_df["destination_balance_error"] = (
    risk_df["oldbalanceDest"]
    + risk_df["amount"]
    - risk_df["newbalanceDest"]
).abs()

In [17]:
fraud_features_comparison = (
    risk_df.groupby("isFraud")
    .agg(
        transactions_count=("isFraud", "size"),
        median_amount=("amount", "median"),
        mean_amount=("amount", "mean"),
        median_balance_ratio=("amount_to_balance_ratio", "median"),
        emptied_account_rate=("emptied_origin_account", "mean"),
        median_origin_error=("origin_balance_error", "median"),
        median_destination_error=("destination_balance_error", "median")
    )
)

fraud_features_comparison["emptied_account_percent"] = (
    fraud_features_comparison["emptied_account_rate"] * 100
).round(2)

fraud_features_comparison.round(2)

,transactions_count,median_amount,mean_amount,median_balance_ratio,emptied_account_rate,median_origin_error,median_destination_error,emptied_account_percent
isFraud,,,,,,,,
0,2762196,171034.46,314115.5,5.78,0.43,144200.83,0.00,42.72
1,8213,441423.44,1467967.3,1.00,0.98,0.00,9511.69,97.55


In [18]:
comparison_by_type = (
    risk_df.groupby(["type", "isFraud"], observed=True)
    .agg(
        transactions_count=("isFraud", "size"),
        median_amount=("amount", "median"),
        median_balance_ratio=("amount_to_balance_ratio", "median"),
        emptied_account_percent=(
            "emptied_origin_account",
            lambda values: values.mean() * 100
        ),
        median_origin_error=("origin_balance_error", "median"),
        median_destination_error=("destination_balance_error", "median")
    )
    .round(2)
)

comparison_by_type

transactions_count  median_amount  median_balance_ratio  \
type     isFraud                                                            
CASH_OUT 0                   2233384      146946.56                  4.78   
         1                      4116      435516.90                  1.00   
TRANSFER 0                    528812      486521.91                 16.23   
         1                      4097      445705.76                  1.00   

                  emptied_account_percent  median_origin_error  \
type     isFraud                                                 
CASH_OUT 0                          42.78            119824.22   
         1                          99.08                 0.00   
TRANSFER 0                          42.50            454033.08   
         1                          96.02                 0.00   

                  median_destination_error  
type     isFraud                            
CASH_OUT 0                            0.00  
         1                            0.00  
TRANSFER 0                            0.00  
         1                       443893.69

By the analysis of dataset, we can surely say that type of fraud operations are CASH_OUT or TRANSFER because of capabilities of output the money. Also, there is a sign of cashout that shows scammers makes the withdrawal of all existing cash in the account and ratio of money is 1. The accounts after the fraud operations have leftovers approximately 0. By the code we counted the median cash for the transactions and the for fraud operations of CASH_OUT it was 435517, while for ordinary's 146947. For TRANSFER it was   486522 for ordinary, 445706 for fraud, that showed us the difference for type to consider the amount by median. Conclusion:
type = TRANSFER or CASH_OUT 
amount ≈ oldbalanceOrg  
amount_to_balance_ratio ≈ 1
newbalanceOrig = 0 
origin_balance_error ≈ 0 

So, by this analysis we can get the necessary data for working in ml perspectives